In [1]:
import pandas as pd
from pathlib import Path

# from posydon.popsyn.binarypopulation import BinaryPopulation
# from posydon.binary_evol.binarystar import BinaryStar
# from posydon.binary_evol.singlestar import SingleStar
from posydon.popsyn.synthetic_population import Population
# from posydon.popsyn.synthetic_population import PopulationRunner
# import astropy.units as u

# import os
# import shutil
# from posydon.config import PATH_TO_POSYDON

# from POSYDONHRDiagramModule import HR_Diagram
# import matplotlib.pyplot as plt
# import random as rand 

from collections import Counter

DataPath = Path().resolve().parent / 'Data'


In [2]:
cols = ['time', 'step_names', 'state', 'event', 'S1_state', 'S2_state', 'S1_mass', 'S2_mass', 'orbital_period']
finCols = [
    'orbital_period_f',
    'eccentricity_f',
    'state_f',

    'S1_state_f',
    'S2_state_f',
    
    'S2_mass_f',
    'S2_log_R_f',
    'S2_log_L_f',


    'S1_mass_f',
    'S1_log_R_f',
    'S1_log_L_f'

 ]

initCols = [
    'orbital_period_i',
    'eccentricity_i',
    'state_i',

    'S2_state_i',
    'S2_mass_i',
    'S2_log_R_i',

    'S1_state_i',
    'S1_mass_i',
    'S1_log_R_i'
 ]

In [3]:
pop = Population(str(DataPath / 'bhSolSubpop.h5'))

In [4]:
len(pop)

39161

In [5]:
# pop.calculate_formation_channels()

In [6]:
Counter(pop.formation_channels['channel'])

Counter({'ZAMS_oRLO1_CC1_oRLO2_CC2_END': 13728,
         'ZAMS_oRLO1_CC1_oRLO2_CC2_maxtime_END': 7867,
         'ZAMS_CC1_oRLO2_CC2_maxtime_END': 4133,
         'ZAMS_CC1_oRLO2_CC2_END': 3981,
         'ZAMS_oRLO1_CC1_oRLO2_oCE2_oMerging2_END': 2854,
         'ZAMS_oRLO1_CC1_END': 1198,
         'ZAMS_CC1_CC2_END': 1131,
         'ZAMS_oRLO1_CC1_oRLO2_oCE2_CC2_END': 1002,
         'ZAMS_CC1_CC2_maxtime_END': 801,
         'ZAMS_CC1_oRLO2_oCE2_oMerging2_END': 375,
         'ZAMS_oRLO1_CC1_oRLO2_oCE2_CC2_maxtime_END': 323,
         'ZAMS_oCE1_CC1_oRLO2_CC2_maxtime_END': 251,
         'ZAMS_oRLO1-contact_CC1_oRLO2_oCE2_oMerging2_END': 247,
         'ZAMS_oRLO1_CC1_oRLO2_oCE2_END': 220,
         'ZAMS_oRLO1_CC1_CC2_END': 184,
         'ZAMS_oRLO1-contact_CC1_END': 141,
         'ZAMS_oRLO1_CC1_CC2_maxtime_END': 121,
         'ZAMS_CC1_END': 86,
         'ZAMS_oCE1_CC1_END': 86,
         'ZAMS_oCE1_CC1_oRLO2_oCE2_oMerging2_END': 74,
         'ZAMS_oCE1_CC1_oRLO2_CC2_END': 55,
         'ZAMS

In [7]:
solBH_oneline_df = pop.oneline.select()
solBH_history_df = pop.history.select()

In [8]:
sol_bhInitMask = (
    ((solBH_history_df['S1_state'] == 'BH') & (solBH_history_df['S2_state'] == 'H-rich_Core_H_burning')) |
    ((solBH_history_df['S2_state'] == 'BH') & (solBH_history_df['S1_state'] == 'H-rich_Core_H_burning'))
) & (solBH_history_df['event'] != 'END')

sol_bhMaskNext = sol_bhInitMask.groupby(level=0).shift(1, fill_value=False)

sol_bhMask = sol_bhInitMask | sol_bhMaskNext

specStatedf = solBH_history_df[sol_bhMask] ### makes a df with the row with the sol-bh and the row after. used for finding the lifespan of the systems 

sol_bhMaskFirst = pd.Series(False, index = sol_bhMask.index)
sol_bhMaskFirst = ~sol_bhMaskFirst.index.duplicated(keep='first')

firstAndsol_bh = sol_bhInitMask | sol_bhMaskFirst

# def isMW(group): ### this is wayyyyy too narrow of a process... it does work though!
#     return (group['time'].iloc[0] < 10e9) & (group['time'].iloc[1] > 10e9)
# results = solBH_history_df[sol_bhMask].groupby(level=0).apply(isMW)

def findStatDur(group):
        return (group['time'].iloc[1] - group['time'].iloc[0])


In [9]:
solBH_history_df[firstAndsol_bh][cols]

,time,step_names,state,event,S1_state,S2_state,S1_mass,S2_mass,orbital_period
binary_index,,,,,,,,,
0,1.354058e+10,initial_cond,detached,ZAMS,H-rich_Core_H_burning,H-rich_Core_H_burning,35.594066,20.634703,86.368549
0,1.354658e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,11.742339,20.766340,773.934254
1,9.269310e+08,initial_cond,detached,ZAMS,H-rich_Core_H_burning,H-rich_Core_H_burning,21.815063,21.446482,4.949364
1,9.365578e+08,step_SN,detached,NaN,BH,H-rich_Core_H_burning,2.862360,27.696843,13.464326
1,9.365578e+08,step_detached,initial_RLOF,NaN,BH,H-rich_Core_H_burning,2.862360,27.479589,13.464326
...,...,...,...,...,...,...,...,...,...
39206,9.352825e+09,initial_cond,detached,ZAMS,H-rich_Core_H_burning,H-rich_Core_H_burning,20.373868,8.859054,154.288032
39206,9.362281e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.746892,9.131854,0.941797
39206,9.362281e+09,step_detached,initial_RLOF,NaN,BH,H-rich_Core_H_burning,3.746892,9.135707,0.941797


In [10]:
sol_bh_lifespan_list = specStatedf.groupby(level=0).apply(findStatDur)
sol_bh_FormationTime_List = solBH_history_df[firstAndsol_bh].groupby(level=0).apply(findStatDur)

In [11]:
sol_bh_lifespan_list.describe()

count    3.916100e+04
mean     7.712614e+06
std      2.035345e+07
min      0.000000e+00
25%      1.116906e+06
50%      3.391723e+06
75%      8.622863e+06
max      1.373802e+09
dtype: float64

In [12]:
sol_bh_FormationTime_List.describe()

count    3.916100e+04
mean     5.937467e+06
std      2.474277e+07
min      3.200919e+06
25%      4.150988e+06
50%      5.172376e+06
75%      6.617883e+06
max      2.467654e+09
dtype: float64

In [ ]:
tolerance = 1e9

sol_bhMaskMWdur = ((
    ((solBH_history_df['S1_state'] == 'BH') & (solBH_history_df['S2_state'] == 'H-rich_Core_H_burning')) |
    ((solBH_history_df['S2_state'] == 'BH') & (solBH_history_df['S1_state'] == 'H-rich_Core_H_burning'))
    ) & (solBH_history_df['event'] != 'END')
        & (solBH_history_df['time'] > 10e9 - tolerance)
        & (solBH_history_df['time'] < 10e9 + tolerance)
    & (solBH_history_df['state'] == 'detached')
)

In [31]:
lowPorbSolBH = solBH_history_df[sol_bhMaskMWdur][solBH_history_df[sol_bhMaskMWdur]['orbital_period'] < 3]

In [32]:
lowPorbSolBH

,state,event,time,separation,orbital_period,eccentricity,rl_relative_overflow_1,rl_relative_overflow_2,lg_mtransfer_rate,step_names,...,S2_conv_env_turnover_time_l_b,S2_conv_env_turnover_time_l_t,S2_envelope_binding_energy,S2_mass_conv_reg_fortides,S2_thickness_conv_reg_fortides,S2_radius_conv_reg_fortides,S2_lambda_CE_1cent,S2_lambda_CE_10cent,S2_lambda_CE_30cent,S2_lambda_CE_pure_He_star_10cent
binary_index,,,,,,,,,,,,,,,,,,,,,
247,detached,NaN,9.005453e+09,17.260173,2.597624,0.731268,NaN,NaN,NaN,step_SN,...,0.000000e+00,0.000000e+00,-4.920224e+49,1.751981e-01,0.053072,1.714441,1.500545,1.500545,1.500545,1.506050
582,detached,NaN,1.095540e+10,13.974633,1.460274,0.081986,NaN,NaN,NaN,step_SN,...,2.068964e+00,-1.000000e+99,-7.742313e+49,3.215535e-07,0.028851,4.691799,1.310448,1.310448,1.310448,1.310448
703,detached,NaN,1.071638e+10,25.639921,2.895749,0.258043,NaN,NaN,NaN,step_SN,...,-7.814006e+98,-7.814006e+98,-1.570231e+50,6.267881e-01,0.248905,6.461539,1.492789,1.492789,1.289354,1.497068
2029,detached,NaN,9.467351e+09,22.757058,2.399101,0.413139,NaN,NaN,NaN,step_SN,...,-9.740103e+98,-1.000000e+99,-1.554792e+50,1.760771e-06,0.156886,7.867052,1.472960,1.472960,1.472960,1.473763
2086,detached,NaN,9.179265e+09,17.787168,2.581370,0.394472,NaN,NaN,NaN,step_SN,...,-1.737363e+91,-1.000000e+99,-5.697924e+49,1.409217e-07,0.012028,3.940619,1.266923,1.266923,1.266923,1.266923
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37932,detached,NaN,9.790867e+09,15.224047,2.092048,0.235246,NaN,NaN,NaN,step_SN,...,-3.106424e+91,-1.000000e+99,-4.582246e+49,1.308894e-09,0.005746,3.376193,1.264025,1.264025,1.264025,1.264025
38056,detached,NaN,9.258820e+09,12.733134,1.565752,0.180059,NaN,NaN,NaN,step_SN,...,-2.060157e+91,-1.000000e+99,-4.935613e+49,1.369782e-09,0.006161,3.554741,1.268890,1.268890,1.268890,1.268890
38116,detached,NaN,9.836158e+09,25.862374,2.993418,0.351462,NaN,NaN,NaN,step_SN,...,-7.053111e+98,-7.053111e+98,-1.500950e+50,1.097844e-01,0.119667,5.947732,1.518408,1.518408,1.518408,1.520201


In [33]:
lowPorbSolBH.to_csv(DataPath / 'threeDayPorb_solBHPairRows.csv')

In [16]:
pop.export_selection(lowPorbSolBH.index.to_list(), str(DataPath / 'bhSol_10Gyr_LowPorb_Subpop.h5'), append=False, overwrite=True)

In [27]:
FiveDPorbSolBH = solBH_history_df[sol_bhMaskMWdur][solBH_history_df[sol_bhMaskMWdur]['orbital_period'] < 5]

In [35]:
FiveDPorbSolBH[cols]

,time,step_names,state,event,S1_state,S2_state,S1_mass,S2_mass,orbital_period
binary_index,,,,,,,,,
30,1.046013e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,9.619724,28.913699,3.789623
247,9.005453e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,2.720699,7.515792,2.597624
582,1.095540e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,5.858426,11.333348,1.460274
703,1.071638e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.925841,23.076179,2.895749
984,1.089803e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,2.627471,8.534797,4.370621
...,...,...,...,...,...,...,...,...,...
39010,1.069579e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.204625,15.702403,2.474185
39011,1.015165e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.149919,16.967927,4.037030
39062,9.813266e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.077906,6.586934,4.465087


In [ ]:
pop.export_selection(FiveDPorbSolBH.index.to_list(), str(DataPath / 'bhSol_10Gyr_FiveDay_Subpop.h5'), append=False, overwrite=True)

In [30]:
filtPop = Population(str(DataPath / 'bhSol_10Gyr_LowPorb_Subpop.h5'))

In [18]:
len(filtPop)

70

In [19]:
filtPop.history.select()[cols]

,time,step_names,state,event,S1_state,S2_state,S1_mass,S2_mass,orbital_period
binary_index,,,,,,,,,
0,8.995911e+09,initial_cond,detached,ZAMS,H-rich_Core_H_burning,H-rich_Core_H_burning,21.649254,7.025360,13.252380
0,9.005453e+09,step_HMS_HMS,detached,CC1,stripped_He_Core_C_depleted,H-rich_Core_H_burning,6.748006,7.515792,2.207866
0,9.005453e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,2.720699,7.515792,2.597624
0,9.005453e+09,step_detached,initial_RLOF,NaN,BH,H-rich_Core_H_burning,2.720699,7.536774,2.597624
0,9.005453e+09,step_end,initial_RLOF,END,BH,H-rich_Core_H_burning,2.720699,7.536774,2.597624
...,...,...,...,...,...,...,...,...,...
69,9.362273e+09,step_CE,detached,NaN,stripped_He_non_burning,H-rich_Core_H_burning,7.457188,9.061668,1.493836
69,9.362281e+09,step_detached,detached,CC1,stripped_He_Core_He_depleted,H-rich_Core_H_burning,7.396282,9.131854,1.514407
69,9.362281e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.746892,9.131854,0.941797
